# Nhiệm vụ 13 — Tổng kết M2 và kiểm tra bàn giao M3
Notebook kiểm tra quyết định mô hình, trạng thái kiểm toán Task 12 và các file bàn giao hiện có. Không tự xác nhận PASS nếu thiếu bằng chứng.

In [1]:
import json
from pathlib import Path
from datetime import datetime
import hashlib
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
while not (ROOT / "M2" / "Ke_hoach_M2_Phan_cum_co_phieu.md").is_file():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Khong tim thay thu muc goc repo M2.")
    ROOT = ROOT.parent

print("Repo root:", ROOT)


Repo root: D:\Clone_M2\Risk-Adjusted-Momentum-Clustering


In [2]:
from pathlib import Path
import json

decision_path = ROOT / "M2/artifacts/m2-evaluation/final_method_decision.json"
audit_path = ROOT / "M2/artifacts/m2-task12-verify-20261004-232543/verify_summary.json"

decision = {}
audit = {}

if decision_path.exists():
    decision = json.loads(decision_path.read_text(encoding="utf-8-sig"))

if audit_path.exists():
    audit = json.loads(audit_path.read_text(encoding="utf-8-sig"))

print("=== QUYET DINH MO HINH ===")
print(json.dumps(decision, ensure_ascii=False, indent=2))

print("\n=== KIEM TOAN NHIEM VU 12 ===")
print(json.dumps(audit, ensure_ascii=False, indent=2))


=== QUYET DINH MO HINH ===
{
  "decision_timestamp": "2026-10-04T08:01:06+00:00",
  "selected_method": "KMeans_Baseline",
  "comparator_methods": [
    "Ward_Hierarchical",
    "PCA_KMeans"
  ],
  "global_k": 2,
  "decision_criteria_rank": [
    "1. Median Silhouette (Primary)",
    "2. Median Davies-Bouldin (Confirmation / Tie-breaker)",
    "3. Cluster Balance (Sanity check)",
    "4. Temporal Stability (ARI / Migration)",
    "5. Occam's Razor (Simplicity & direct interpretability)"
  ],
  "selection_rationale": "KMeans_Baseline ???c ch?n sau khi c? ba ph??ng ?n v??t sanity check. So v?i Ward_Hierarchical, KMeans c? c?ng Median Silhouette=0.755722 nh?ng DB th?p h?n (0.521899 so v?i 0.534333), ARI cao h?n (0.798251 so v?i 0.537278) v? Migration th?p h?n (0.014382 so v?i 0.037716). So v?i PCA_KMeans, PCA c? Median Silhouette cao h?n (0.771666 so v?i 0.755722) v? DB th?p h?n, nh?ng ch?nh l?ch Silhouette ch? 0.015945, nh? h?n ng??ng Occam 0.03; Balance v? b?n ch? s? temporal c?a hai ph?

In [3]:
required_files = [
    "M2/artifacts/m2-final-handoff-v1/m2_cluster_labels_for_m3.csv",
    "M2/artifacts/m2-final-handoff-v1/m2_cluster_profiles_for_m3.csv",
    "M2/artifacts/m2-final-handoff-v1/m2_transition_turnover_reference.csv",
    "M2/artifacts/m2-final-handoff-v1/m2_to_m3_handoff_manifest.json",
    "M2/reports/Bao_cao_M2_Nhiem_vu_12_Verify.md",
]

checks = []
for rel in required_files:
    p = ROOT / rel
    checks.append({
        "file": rel,
        "exists": p.is_file(),
        "size_bytes": p.stat().st_size if p.is_file() else None
    })

checks_df = pd.DataFrame(checks)
display(checks_df)


,file,exists,size_bytes
0,M2/artifacts/m2-final-handoff-v1/m2_cluster_la...,True,1470322
1,M2/artifacts/m2-final-handoff-v1/m2_cluster_pr...,True,28177
2,M2/artifacts/m2-final-handoff-v1/m2_transition...,True,8099
3,M2/artifacts/m2-final-handoff-v1/m2_to_m3_hand...,True,2845
4,M2/reports/Bao_cao_M2_Nhiem_vu_12_Verify.md,True,994


In [4]:
# Kiem tra du lieu va dieu kien ban giao
decision_ok = (
    decision.get("selected_method") == "KMeans_Baseline"
    and decision.get("global_k") == 2
)

audit_status = str(audit.get("status", "MISSING")).upper()
audit_ok = audit_status in {"PASS", "PASSED", "SUCCESS"}

files_ok = bool(checks_df["exists"].all())

handoff_status = "READY_FOR_REVIEW" if decision_ok and audit_ok and files_ok else "BLOCKED"

print("Decision valid:", decision_ok)
print("Task 12 audit status:", audit_status)
print("Required files exist:", files_ok)
print("Task 13 status:", handoff_status)


Decision valid: True
Task 12 audit status: BLOCKED
Required files exist: True
Task 13 status: BLOCKED


In [5]:
# Luu ket qua kiem tra Nhiem vu 13
output_dir = ROOT / "M2/artifacts/m2-task13-handoff-check"
output_dir.mkdir(parents=True, exist_ok=True)

checks_df.to_csv(output_dir / "required_files_check.csv", index=False, encoding="utf-8-sig")

result = {
    "task": "M2_TASK_13_HANDOFF",
    "generated_at": datetime.now().astimezone().isoformat(),
    "status": handoff_status,
    "decision_ok": bool(decision_ok),
    "task12_audit_status": audit_status,
    "required_files_ok": bool(files_ok),
    "selected_method": decision.get("selected_method"),
    "global_k": decision.get("global_k"),
    "note": "Chi ban giao chinh thuc khi tat ca dieu kien bat buoc duoc xac minh."
}

(output_dir / "handoff_check_summary.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

display(Markdown(f"## Kết quả kiểm tra Nhiệm vụ 13\n\n**Trạng thái:** `{handoff_status}`"))
print("Saved:", output_dir)


## Kết quả kiểm tra Nhiệm vụ 13

**Trạng thái:** `BLOCKED`

Saved: D:\Clone_M2\Risk-Adjusted-Momentum-Clustering\M2\artifacts\m2-task13-handoff-check
